# Random Forest from Scratch

Random Forest is a powerful ensemble method that combines multiple decision trees to achieve superior predictive performance. In this notebook, we'll implement Random Forest from scratch, exploring both the theoretical foundations and practical implementation details.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
import seaborn as sns
sns.set_style("whitegrid")

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## Section 1: Why Single Trees Fail

Decision trees are powerful models, but they suffer from high variance, making them prone to overfitting. This instability means that small changes in the training data can lead to dramatically different trees.

Let's first visualize this issue by training multiple trees on slightly different subsets of the same dataset:

In [ ]:
# Generate a dataset
X, y = make_classification(n_samples=500, n_features=2, n_redundant=0, 
                          n_informative=2, n_clusters_per_class=1, 
                          random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
# Define utility functions for tree implementation
def gini_impurity(y):
    """Calculate Gini impurity for a given array of labels."""
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    gini = 1 - np.sum(probabilities ** 2)
    return gini

def entropy(y):
    """Calculate entropy for a given array of labels."""
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    # Add small epsilon to avoid log(0)
    probabilities = np.clip(probabilities, 1e-15, 1.0)
    entropy_val = -np.sum(probabilities * np.log2(probabilities))
    return entropy_val

def information_gain(parent, left_child, right_child, criterion='gini'):
    """Calculate information gain from splitting parent into left and right children."""
    if criterion == 'gini':
        score_func = gini_impurity
    else:
        score_func = entropy
        
    parent_score = score_func(parent)
    
    # Calculate weighted scores for children
    total_samples = len(left_child) + len(right_child)
    left_weight = len(left_child) / total_samples
    right_weight = len(right_child) / total_samples
    
    weighted_score = (left_weight * score_func(left_child) + 
                     right_weight * score_func(right_child))
    
    return parent_score - weighted_score

In [ ]:
class DecisionTreeNode:
    def __init__(self):
        self.feature_idx = None
        self.threshold = None
        self.left = None
        self.right = None
        self.value = None  # For leaf nodes
        self.samples = None
        self.impurity = None
        self.n_classes = None

In [ ]:
class DecisionTreeClassifier:
    def __init__(self, max_depth=None, min_samples_split=2, min_samples_leaf=1, 
                 max_features=None, criterion='gini', random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        if random_state is not None:
            np.random.seed(random_state)
        
        self.root = None
        self.n_classes = None
        self.n_features = None
        
    def _best_split(self, X, y):
        """Find the best split for the current node."""
        best_gain = -1
        best_feature_idx = None
        best_threshold = None
        
        n_features = X.shape[1]
        
        # Consider only a random subset of features if max_features is set
        if self.max_features is not None:
            feature_indices = np.random.choice(n_features, 
                                            size=min(self.max_features, n_features), 
                                            replace=False)
        else:
            feature_indices = range(n_features)
        
        for feature_idx in feature_indices:
            thresholds = np.unique(X[:, feature_idx])
            
            for threshold in thresholds:
                left_mask = X[:, feature_idx] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples_leaf or np.sum(right_mask) < self.min_samples_leaf:
                    continue
                
                gain = information_gain(y, y[left_mask], y[right_mask], self.criterion)
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature_idx = feature_idx
                    best_threshold = threshold
        
        return best_feature_idx, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        """Recursively build the decision tree."""
        n_samples, n_features = X.shape
        n_classes = len(np.unique(y))
        
        # Create a node
        node = DecisionTreeNode()
        node.samples = n_samples
        node.n_classes = n_classes
        node.impurity = gini_impurity(y) if self.criterion == 'gini' else entropy(y)
        
        # Stopping criteria
        if (self.max_depth is not None and depth >= self.max_depth) or \
           n_samples < self.min_samples_split or \
           n_classes == 1:
            # Create leaf node
            classes, counts = np.unique(y, return_counts=True)
            node.value = classes[np.argmax(counts)]
            return node
        
        # Find best split
        feature_idx, threshold, gain = self._best_split(X, y)
        
        if feature_idx is None or gain <= 0:
            # No good split found, make leaf
            classes, counts = np.unique(y, return_counts=True)
            node.value = classes[np.argmax(counts)]
            return node
        
        # Make split
        node.feature_idx = feature_idx
        node.threshold = threshold
        
        left_mask = X[:, feature_idx] <= threshold
        right_mask = ~left_mask
        
        # Recursively build left and right subtrees
        node.left = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        node.right = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return node
    
    def fit(self, X, y):
        """Fit the decision tree to the training data."""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        self.root = self._build_tree(X, y)
        return self
    
    def _predict_sample(self, x, node):
        """Predict a single sample using the trained tree."""
        if node.value is not None:  # Leaf node
            return node.value
        
        if x[node.feature_idx] <= node.threshold:
            return self._predict_sample(x, node.left)
        else:
            return self._predict_sample(x, node.right)
    
    def predict(self, X):
        """Predict class labels for samples in X."""
        predictions = []
        for x in X:
            pred = self._predict_sample(x, self.root)
            predictions.append(pred)
        return np.array(predictions)
    
    def get_depth(self, node=None):
        """Get the depth of the tree."""
        if node is None:
            node = self.root
        if node.value is not None:  # Leaf node
            return 0
        return 1 + max(self.get_depth(node.left), self.get_depth(node.right))
    
    def count_nodes(self, node=None):
        """Count the number of nodes in the tree."""
        if node is None:
            node = self.root
        if node.value is not None:  # Leaf node
            return 1
        return 1 + self.count_nodes(node.left) + self.count_nodes(node.right)

In [ ]:
# Demonstrate tree instability
print("Demonstrating Tree Instability")
print("="*35)

# Train multiple trees on different random samples
for i in range(5):
    # Create a random subsample
    indices = np.random.choice(len(X_train), size=int(0.8 * len(X_train)), replace=False)
    X_subset = X_train[indices]
    y_subset = y_train[indices]
    
    tree = DecisionTreeClassifier(random_state=42+i)
    tree.fit(X_subset, y_subset)
    
    train_pred = tree.predict(X_train)
    test_pred = tree.predict(X_test)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    
    print(f"Tree {i+1} - Train Acc: {train_acc:.3f}, Test Acc: {test_acc:.3f}, Depth: {tree.get_depth()}")

As we can see, each tree has significantly different performance and structure despite being trained on similar data. This high variance is problematic for reliable predictions.

## Section 2: Bagging - Bootstrap Aggregating

Bagging (Bootstrap Aggregating) is a technique that reduces variance by training multiple models on different bootstrap samples of the training data and combining their predictions through averaging (for regression) or majority voting (for classification).

Mathematically, for a classification problem with $B$ bootstrap samples, the bagged predictor is:

$$\hat{f}_{bag}(x) = \arg\max_c \sum_{b=1}^{B} I(\hat{f}^{*b}(x) = c)$$

Where $\hat{f}^{*b}(x)$ is the prediction from the model trained on the $b$-th bootstrap sample.

For regression:

$$\hat{f}_{bag}(x) = \frac{1}{B} \sum_{b=1}^{B} \hat{f}^{*b}(x)$$

The key insight is that if we have $B$ independent predictors, each with variance $\sigma^2$, the variance of the average predictor is:

$$\text{Var}(\bar{X}) = \frac{\sigma^2}{B} + \frac{B-1}{B}\rho\sigma^2$$

Where $\rho$ is the average correlation between the predictors. As $B \to \infty$, the variance approaches $\rho\sigma^2$. Thus, bagging is most effective when individual predictors are uncorrelated.

In [ ]:
def bootstrap_sample(X, y, random_state=None):
    """Generate a bootstrap sample from the dataset."""
    if random_state is not None:
        np.random.seed(random_state)
    
    n_samples = X.shape[0]
    indices = np.random.choice(n_samples, size=n_samples, replace=True)
    return X[indices], y[indices]

In [ ]:
# Implement a simple bagging classifier
class BaggingClassifier:
    def __init__(self, base_estimator=None, n_estimators=10, random_state=None):
        self.base_estimator = base_estimator if base_estimator is not None else DecisionTreeClassifier()
        self.n_estimators = n_estimators
        self.random_state = random_state
        if random_state is not None:
            np.random.seed(random_state)
        
        self.estimators = []
        
    def fit(self, X, y):
        """Fit the bagging ensemble."""
        self.estimators = []
        
        for i in range(self.n_estimators):
            # Generate bootstrap sample
            X_boot, y_boot = bootstrap_sample(X, y, random_state=self.random_state+i if self.random_state is not None else None)
            
            # Fit estimator on bootstrap sample
            estimator = self.base_estimator.__class__(**self.base_estimator.__dict__)
            estimator.fit(X_boot, y_boot)
            self.estimators.append(estimator)
        
        return self
    
    def predict(self, X):
        """Make predictions using majority voting."""
        predictions = np.zeros((len(X), self.n_estimators))
        
        for i, estimator in enumerate(self.estimators):
            predictions[:, i] = estimator.predict(X)
        
        # Majority voting
        final_predictions = []
        for sample_preds in predictions:
            unique_vals, counts = np.unique(sample_preds, return_counts=True)
            final_predictions.append(unique_vals[np.argmax(counts)])
        
        return np.array(final_predictions)
    
    def predict_proba(self, X):
        """Predict class probabilities."""
        predictions = np.zeros((len(X), self.n_estimators))
        
        for i, estimator in enumerate(self.estimators):
            predictions[:, i] = estimator.predict(X)
        
        # Calculate probability for each class
        classes = np.unique(predictions)
        probas = np.zeros((len(X), len(classes)))
        
        for i, cls in enumerate(classes):
            probas[:, i] = np.mean(predictions == cls, axis=1)
        
        return probas

In [ ]:
# Compare single tree vs bagging
print("Comparing Single Tree vs Bagging")
print("="*35)

# Single tree
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
single_train_pred = single_tree.predict(X_train)
single_test_pred = single_tree.predict(X_test)
single_train_acc = accuracy_score(y_train, single_train_pred)
single_test_acc = accuracy_score(y_test, single_test_pred)

print(f"Single Tree - Train Acc: {single_train_acc:.3f}, Test Acc: {single_test_acc:.3f}")

# Bagging
bagging = BaggingClassifier(base_estimator=DecisionTreeClassifier(max_depth=5), 
                           n_estimators=10, random_state=42)
bagging.fit(X_train, y_train)
bagging_train_pred = bagging.predict(X_train)
bagging_test_pred = bagging.predict(X_test)
bagging_train_acc = accuracy_score(y_train, bagging_train_pred)
bagging_test_acc = accuracy_score(y_test, bagging_test_pred)

print(f"Bagging (10 trees) - Train Acc: {bagging_train_acc:.3f}, Test Acc: {bagging_test_acc:.3f}")

## Section 3: Feature Randomness - The Key Innovation of Random Forest

While bagging helps reduce variance, Random Forest introduces an additional source of randomness: at each split in the tree, only a random subset of features is considered. This decorrelates the trees, making the ensemble even more robust.

For a dataset with $m$ features, at each split, Random Forest considers only $m_{try}$ features chosen at random, where $m_{try}$ is typically $\sqrt{m}$ for classification and $m/3$ for regression.

This additional randomness forces the trees to be more diverse, reducing the correlation between them and improving the overall ensemble performance.

In [ ]:
class RandomForestClassifier:
    def __init__(self, n_estimators=10, max_depth=None, min_samples_split=2, 
                 min_samples_leaf=1, max_features='sqrt', criterion='gini', 
                 random_state=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.criterion = criterion
        self.random_state = random_state
        if random_state is not None:
            np.random.seed(random_state)
        
        self.estimators = []
        self.n_classes = None
        self.n_features = None
        
    def _get_max_features(self, n_features):
        """Calculate the number of features to consider at each split."""
        if self.max_features == 'sqrt':
            return int(np.sqrt(n_features))
        elif self.max_features == 'log2':
            return int(np.log2(n_features))
        elif isinstance(self.max_features, int):
            return min(self.max_features, n_features)
        elif isinstance(self.max_features, float):
            return int(self.max_features * n_features)
        else:
            return n_features
    
    def fit(self, X, y):
        """Fit the random forest to the training data."""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        n_max_features = self._get_max_features(self.n_features)
        
        self.estimators = []
        
        for i in range(self.n_estimators):
            # Generate bootstrap sample
            X_boot, y_boot = bootstrap_sample(X, y, random_state=self.random_state+i if self.random_state is not None else None)
            
            # Create a decision tree with feature randomness
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=n_max_features,
                criterion=self.criterion,
                random_state=self.random_state+i if self.random_state is not None else None
            )
            
            tree.fit(X_boot, y_boot)
            self.estimators.append(tree)
        
        return self
    
    def predict(self, X):
        """Make predictions using majority voting."""
        predictions = np.zeros((len(X), self.n_estimators))
        
        for i, estimator in enumerate(self.estimators):
            predictions[:, i] = estimator.predict(X)
        
        # Majority voting
        final_predictions = []
        for sample_preds in predictions:
            unique_vals, counts = np.unique(sample_preds, return_counts=True)
            final_predictions.append(unique_vals[np.argmax(counts)])
        
        return np.array(final_predictions)
    
    def predict_proba(self, X):
        """Predict class probabilities."""
        predictions = np.zeros((len(X), self.n_estimators))
        
        for i, estimator in enumerate(self.estimators):
            predictions[:, i] = estimator.predict(X)
        
        # Calculate probability for each class
        classes = np.unique(predictions)
        probas = np.zeros((len(X), len(classes)))
        
        for i, cls in enumerate(classes):
            probas[:, i] = np.mean(predictions == cls, axis=1)
        
        return probas
    
    def feature_importances(self):
        """Calculate feature importances based on mean decrease in impurity."""
        importances = np.zeros(self.n_features)
        
        for tree in self.estimators:
            # Calculate importances for this tree
            tree_importances = self._tree_feature_importance(tree.root, tree.n_features)
            importances += tree_importances
        
        # Normalize
        importances /= self.n_estimators
        return importances
    
    def _tree_feature_importance(self, node, n_features, importances=None):
        """Recursively calculate feature importances for a single tree."""
        if importances is None:
            importances = np.zeros(n_features)
        
        if node.value is not None:  # Leaf node
            return importances
        
        # Calculate importance as decrease in impurity weighted by node size
        if node.left.value is not None and node.right.value is not None:  # Both children are leaves
            # For internal node, calculate the improvement
            improvement = (node.impurity - 
                          (node.left.samples * node.left.impurity + 
                           node.right.samples * node.right.impurity) / node.samples)
            importances[node.feature_idx] += improvement
        
        # Recursively calculate for children
        self._tree_feature_importance(node.left, n_features, importances)
        self._tree_feature_importance(node.right, n_features, importances)
        
        return importances

## Section 4: Random Forest Implementation

Now let's implement the complete Random Forest classifier and test it on our dataset:

In [ ]:
# Train Random Forest
rf = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# Make predictions
rf_train_pred = rf.predict(X_train)
rf_test_pred = rf.predict(X_test)

# Calculate accuracies
rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print(f"Random Forest - Train Acc: {rf_train_acc:.3f}, Test Acc: {rf_test_acc:.3f}")

## Section 5: Out-of-Bag (OOB) Error Estimation

One of the elegant features of Random Forest is the ability to estimate the generalization error without requiring a separate validation set. Since each tree is trained on a bootstrap sample (which contains approximately 63.2% of the original data), the remaining 36.8% of samples not used for training that tree can be used as a validation set.

These samples are called Out-of-Bag (OOB) samples. The OOB error is calculated as follows:
1. For each sample in the original dataset, identify which trees had that sample in their OOB set
2. Average the predictions from those trees
3. Calculate the error based on these averaged predictions

The OOB error provides an unbiased estimate of the test error.

In [ ]:
class RandomForestClassifierWithOOB(RandomForestClassifier):
    def __init__(self, n_estimators=10, max_depth=None, min_samples_split=2, 
                 min_samples_leaf=1, max_features='sqrt', criterion='gini', 
                 random_state=None):
        super().__init__(n_estimators, max_depth, min_samples_split, 
                         min_samples_leaf, max_features, criterion, random_state)
        self.oob_score_ = None
        
    def fit(self, X, y):
        """Fit the random forest and calculate OOB score."""
        self.n_classes = len(np.unique(y))
        self.n_features = X.shape[1]
        n_max_features = self._get_max_features(self.n_features)
        
        self.estimators = []
        
        # Track which samples are OOB for each tree
        oob_predictions = [[] for _ in range(len(X))]
        oob_counts = np.zeros(len(X))
        
        for i in range(self.n_estimators):
            # Generate bootstrap sample
            boot_indices = np.random.choice(len(X), size=len(X), replace=True)
            X_boot = X[boot_indices]
            y_boot = y[boot_indices]
            
            # Identify OOB samples for this tree
            oob_mask = np.ones(len(X), dtype=bool)
            for idx in boot_indices:
                oob_mask[idx] = False
            
            # Create and train the tree
            tree = DecisionTreeClassifier(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                min_samples_leaf=self.min_samples_leaf,
                max_features=n_max_features,
                criterion=self.criterion,
                random_state=self.random_state+i if self.random_state is not None else None
            )
            
            tree.fit(X_boot, y_boot)
            self.estimators.append(tree)
            
            # Get predictions for OOB samples
            oob_indices = np.where(oob_mask)[0]
            if len(oob_indices) > 0:
                oob_X = X[oob_indices]
                oob_pred = tree.predict(oob_X)
                
                for j, idx in enumerate(oob_indices):
                    oob_predictions[idx].append(oob_pred[j])
                    oob_counts[idx] += 1
        
        # Calculate OOB score
        oob_correct = 0
        oob_total = 0
        
        for i in range(len(X)):
            if oob_counts[i] > 0:  # Sample was OOB for at least one tree
                # Majority vote among OOB predictions
                preds = oob_predictions[i]
                unique_preds, counts = np.unique(preds, return_counts=True)
                final_pred = unique_preds[np.argmax(counts)]
                
                if final_pred == y[i]:
                    oob_correct += 1
                oob_total += 1
        
        if oob_total > 0:
            self.oob_score_ = oob_correct / oob_total
        else:
            self.oob_score_ = 0.0
        
        return self
    
    def get_oob_score(self):
        """Return the out-of-bag score."""
        return self.oob_score_

In [ ]:
# Train Random Forest with OOB estimation
rf_oob = RandomForestClassifierWithOOB(n_estimators=10, max_depth=5, random_state=42)
rf_oob.fit(X_train, y_train)

# Get OOB score
oob_score = rf_oob.get_oob_score()
print(f"Out-of-Bag Score: {oob_score:.3f}")

# Compare with actual test score
rf_test_pred = rf_oob.predict(X_test)
rf_test_acc = accuracy_score(y_test, rf_test_pred)
print(f"Test Accuracy: {rf_test_acc:.3f}")
print(f"Difference (|Test - OOB|): {abs(rf_test_acc - oob_score):.3f}")

## Section 6: Comprehensive Experiments

Now let's conduct experiments comparing single trees, pruned trees, and Random Forest:

In [ ]:
# Generate a larger dataset for more meaningful experiments
X_exp, y_exp = make_classification(n_samples=1000, n_features=10, n_informative=5, 
                                   n_redundant=2, n_clusters_per_class=1, 
                                   random_state=42)

X_train_exp, X_test_exp, y_train_exp, y_test_exp = train_test_split(
    X_exp, y_exp, test_size=0.3, random_state=42)

In [ ]:
# Compare different models
print("Model Comparison")
print("="*50)

# Single deep tree (likely to overfit)
single_deep = DecisionTreeClassifier(random_state=42)
single_deep.fit(X_train_exp, y_train_exp)
single_deep_train_acc = accuracy_score(y_train_exp, single_deep.predict(X_train_exp))
single_deep_test_acc = accuracy_score(y_test_exp, single_deep.predict(X_test_exp))

# Single pruned tree
single_pruned = DecisionTreeClassifier(max_depth=5, random_state=42)
single_pruned.fit(X_train_exp, y_train_exp)
single_pruned_train_acc = accuracy_score(y_train_exp, single_pruned.predict(X_train_exp))
single_pruned_test_acc = accuracy_score(y_test_exp, single_pruned.predict(X_test_exp))

# Random Forest
rf_exp = RandomForestClassifier(n_estimators=10, max_depth=5, random_state=42)
rf_exp.fit(X_train_exp, y_train_exp)
rf_train_acc = accuracy_score(y_train_exp, rf_exp.predict(X_train_exp))
rf_test_acc = accuracy_score(y_test_exp, rf_exp.predict(X_test_exp))

print(f"Single Deep Tree - Train: {single_deep_train_acc:.3f}, Test: {single_deep_test_acc:.3f}, Gap: {single_deep_train_acc - single_deep_test_acc:.3f}")
print(f"Single Pruned Tree - Train: {single_pruned_train_acc:.3f}, Test: {single_pruned_test_acc:.3f}, Gap: {single_pruned_train_acc - single_pruned_test_acc:.3f}")
print(f"Random Forest - Train: {rf_train_acc:.3f}, Test: {rf_test_acc:.3f}, Gap: {rf_train_acc - rf_test_acc:.3f}")

In [ ]:
# Plot comparison
models = ['Single Deep', 'Single Pruned', 'Random Forest']
train_scores = [single_deep_train_acc, single_pruned_train_acc, rf_train_acc]
test_scores = [single_deep_test_acc, single_pruned_test_acc, rf_test_acc]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, train_scores, width, label='Train Accuracy', alpha=0.8)
bars2 = ax.bar(x + width/2, test_scores, width, label='Test Accuracy', alpha=0.8)

ax.set_xlabel('Models')
ax.set_ylabel('Accuracy')
ax.set_title('Model Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom')

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Section 7: Feature Importance Analysis

Random Forest provides a natural way to measure feature importance based on how much each feature contributes to decreasing node impurity across all trees in the forest. The feature importance is calculated as the average decrease in impurity brought by splits on that feature.

In [ ]:
# Calculate and visualize feature importances
rf_imp = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_imp.fit(X_train_exp, y_train_exp)

importances = rf_imp.feature_importances()
std = np.std([tree.feature_importances if hasattr(tree, 'feature_importances') 
              else np.zeros(X_train_exp.shape[1]) for tree in rf_imp.estimators], axis=0)

# Sort feature importances in descending order
indices = np.argsort(importances)[::-1]

# Plot feature importances
plt.figure(figsize=(12, 6))
plt.title("Feature Importances in Random Forest")
plt.bar(range(X_train_exp.shape[1]), importances[indices],
        yerr=std[indices] if len(std) == len(importances) else None,
        align="center")
plt.xticks(range(X_train_exp.shape[1]), [f'Feature {i}' for i in indices], rotation=45)
plt.xlim([-1, X_train_exp.shape[1]])
plt.ylabel('Importance')
plt.xlabel('Features')
plt.tight_layout()
plt.show()

print("Feature ranking:")
for i in range(X_train_exp.shape[1]):
    print(f"{i+1}. Feature {indices[i]} ({importances[indices[i]]:.4f})")

## Section 8: Final Discussion

### Why Random Forest Generalizes Better

Random Forest achieves superior generalization through several mechanisms:

1. **Bootstrap Aggregation**: Reduces variance by averaging predictions from multiple models trained on different data samples
2. **Feature Randomness**: Decorrelates trees by considering only a random subset of features at each split, further reducing ensemble variance
3. **Bias-Variance Tradeoff**: While individual trees might have low bias, the ensemble approach significantly reduces variance without substantially increasing bias
4. **Robustness**: Less sensitive to outliers and noise compared to single trees

### When Random Forest Might Fail

Despite its strengths, Random Forest can struggle in certain scenarios:

1. **Highly Non-stationary Data**: When the underlying data distribution changes over time
2. **Very High Dimensional Sparse Data**: Such as text data with TF-IDF features, where linear models might perform better
3. **Need for Interpretability**: Individual decision rules are harder to extract from an ensemble
4. **Computational Constraints**: Training and prediction can be slower than simpler models

### Computational Complexity

For a Random Forest with $B$ trees, each of maximum depth $d$, and $m$ features:

- **Training Complexity**: $O(B \cdot n \cdot d \cdot m)$ where $n$ is the number of samples
- **Prediction Complexity**: $O(B \cdot d)$ for each prediction

However, training and prediction can be easily parallelized across trees, making Random Forest highly scalable to multi-core systems.

### Conclusion

Random Forest is a robust, versatile ensemble method that effectively balances bias and variance while providing additional benefits like feature importance and OOB error estimation. Its success stems from the combination of bagging and feature randomness, which work synergistically to create diverse, accurate individual learners that combine to form a powerful ensemble.